# MNIST Image Denoising using Convolutional Autoencoder

Week 6 Assessment project. The model removes noise from MNIST digit images using a convolutional autoencoder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
OUTPUT_DIR = Path("outputs")
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load MNIST Dataset

The notebook uses Kaggle CSV files if available in `data/`. Otherwise, it loads the same MNIST dataset from TensorFlow.

In [ ]:
def load_mnist_data():
    kaggle_train = DATA_DIR / "mnist_train.csv"
    kaggle_test = DATA_DIR / "mnist_test.csv"
    alt_train = DATA_DIR / "train.csv"
    alt_test = DATA_DIR / "test.csv"

    if kaggle_train.exists() and kaggle_test.exists():
        train_df = pd.read_csv(kaggle_train)
        test_df = pd.read_csv(kaggle_test)
        x_train = train_df.iloc[:, 1:].values.reshape(-1, 28, 28, 1)
        x_test = test_df.iloc[:, 1:].values.reshape(-1, 28, 28, 1)
        source = "Kaggle CSV files"
    elif alt_train.exists():
        train_df = pd.read_csv(alt_train)
        x_train = train_df.drop(columns=["label"]).values.reshape(-1, 28, 28, 1) if "label" in train_df.columns else train_df.iloc[:, 1:].values.reshape(-1, 28, 28, 1)
        if alt_test.exists():
            test_df = pd.read_csv(alt_test)
            x_test = test_df.drop(columns=["label"]).values.reshape(-1, 28, 28, 1) if "label" in test_df.columns else test_df.values.reshape(-1, 28, 28, 1)
        else:
            x_test = x_train[-10000:]
            x_train = x_train[:-10000]
        source = "Kaggle train.csv/test.csv"
    else:
        (x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
        x_train = x_train.reshape(-1, 28, 28, 1)
        x_test = x_test.reshape(-1, 28, 28, 1)
        source = "TensorFlow MNIST"

    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0
    return x_train, x_test, source

x_train, x_test, source = load_mnist_data()
print("Dataset source:", source)
print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

## 2. Add Gaussian Noise

In [ ]:
def add_noise(images, noise_factor=0.45):
    noisy = images + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=images.shape)
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)

## 3. Visualize Clean and Noisy Images

In [ ]:
plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    plt.subplot(2, 8, i + 9)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
plt.suptitle("Original Images vs Noisy Images")
plt.show()

## 4. Build Convolutional Autoencoder

In [ ]:
def build_autoencoder():
    input_img = layers.Input(shape=(28, 28, 1))

    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input_img)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)
    x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D((2, 2), padding="same")(x)

    x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(encoded)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.UpSampling2D((2, 2))(x)
    decoded = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")(x)

    autoencoder = models.Model(input_img, decoded)
    autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
    return autoencoder

autoencoder = build_autoencoder()
autoencoder.summary()

## 5. Train the Model

In [ ]:
history = autoencoder.fit(
    x_train_noisy,
    x_train,
    epochs=8,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test)
)

## 6. Save Model and Plot Loss

In [ ]:
autoencoder.save("models/mnist_denoising_autoencoder.keras")

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/training_loss.png", dpi=150)
plt.show()

## 7. Predict and Visualize Denoised Output

In [ ]:
denoised_images = autoencoder.predict(x_test_noisy[:10])

n = 10
plt.figure(figsize=(18, 5))
for i in range(n):
    plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title("Original", fontsize=9)
    plt.axis("off")

    plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap="gray")
    plt.title("Noisy", fontsize=9)
    plt.axis("off")

    plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised_images[i].reshape(28, 28), cmap="gray")
    plt.title("Denoised", fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.savefig("outputs/denoising_results.png", dpi=150)
plt.show()

## Conclusion

The autoencoder learns compressed features from noisy handwritten digits and reconstructs cleaner outputs. This proves that autoencoders can be used for image denoising and reconstruction tasks.